In [1]:
"""
QFE-COD Kaggle Gradio Dashboard
================================
Upload results.zip as a Kaggle dataset, then run this script.
Paste each section into separate Kaggle notebook cells.
"""

# ============================================================================
# CELL 1: Setup & Paths
# ============================================================================
import subprocess, sys
from pathlib import Path

def pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])
pip('gradio', 'timm', 'pennylane', 'pennylane-lightning', 'PyWavelets',
    'albumentations', 'einops', 'torchmetrics')

# Direct Kaggle dataset path (uploaded as QFE_Pred)
BASE = Path('/kaggle/input/datasets/dakshethg/qfe-pred')
# Fallback for standard Kaggle dataset path
if not BASE.exists():
    BASE = Path('/kaggle/input/qfe-pred')
CKPT    = BASE / 'checkpoints'
LOG_DIR = BASE / 'logs'
VIZ_DIR = BASE / 'viz'

print('Checkpoints:', list(CKPT.iterdir()) if CKPT.exists() else 'MISSING')
print('Logs:', list(LOG_DIR.iterdir()) if LOG_DIR.exists() else 'MISSING')
print('Viz:', list(VIZ_DIR.iterdir()) if VIZ_DIR.exists() else 'MISSING')

# ============================================================================
# CELL 2: Imports & Config
# ============================================================================
import math, warnings, numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import timm, cv2, gradio as gr
from PIL import Image
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt, matplotlib.cm as cm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import pennylane as qml

warnings.filterwarnings('ignore')
IMG_SIZE = 352
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

val_aug = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2(),
])
print(f'Device: {DEVICE} | Torch: {torch.__version__}')

# ============================================================================
# CELL 3: Model Definitions
# ============================================================================
class DWTDecomp(nn.Module):
    def forward(self, x):
        H, W = x.shape[2], x.shape[3]
        x = x[:,:,:H-H%2,:W-W%2]
        x00,x01,x10,x11 = x[:,:,0::2,0::2],x[:,:,0::2,1::2],x[:,:,1::2,0::2],x[:,:,1::2,1::2]
        return (x00+x01+x10+x11)*0.5,(-x00-x01+x10+x11)*0.5,(-x00+x01-x10+x11)*0.5,(x00-x01-x10+x11)*0.5

class FSA(nn.Module):
    def __init__(self, in_ch):
        super().__init__()
        self.pool=nn.AdaptiveAvgPool2d(1); self.proj=nn.Linear(in_ch,2)
        self.attn=nn.MultiheadAttention(2,2,batch_first=True); self.out_proj=nn.Linear(8,8)
    def forward(self, ll,lh,hl,hh):
        tokens=torch.stack([self.proj(self.pool(b).squeeze(-1).squeeze(-1)) for b in [ll,lh,hl,hh]],dim=1)
        out,_=self.attn(tokens,tokens,tokens)
        return self.out_proj(out.reshape(out.size(0),-1))

class SimpleSSM(nn.Module):
    def __init__(self, d, d_state=16, expand=2):
        super().__init__()
        di=d*expand
        self.in_proj=nn.Linear(d,di*2); self.conv1d=nn.Conv1d(di,di,4,padding=3,groups=di)
        self.out_proj=nn.Linear(di,d); self.norm=nn.LayerNorm(d)
    def forward(self, x):
        r=x; xz=self.in_proj(x); x2,z=xz.chunk(2,dim=-1)
        x2=self.conv1d(x2.transpose(1,2))[:,:,:x.size(1)].transpose(1,2)
        return self.norm(self.out_proj(F.silu(x2)*F.silu(z))+r)

class MambaBlock(nn.Module):
    def __init__(self,c): super().__init__(); self.ssm=SimpleSSM(c)
    def forward(self,x):
        B,C,H,W=x.shape; return self.ssm(x.flatten(2).transpose(1,2)).transpose(1,2).reshape(B,C,H,W)

class PVTv2Backbone(nn.Module):
    def __init__(self, pretrained=False):
        super().__init__()
        self.model=timm.create_model('pvt_v2_b4',pretrained=pretrained,features_only=True,out_indices=(0,1,2,3))
        self.channels=self.model.feature_info.channels()
    def forward(self,x): return self.model(x)

class FPNDecoder(nn.Module):
    def __init__(self, in_channels=(64,128,320,512), out_ch=64):
        super().__init__()
        self.lat=nn.ModuleList([nn.Conv2d(c,out_ch,1) for c in in_channels])
        self.mamba=nn.ModuleList([MambaBlock(out_ch) for _ in in_channels])
        self.fuse=nn.ModuleList([nn.Sequential(nn.Conv2d(out_ch,out_ch,3,padding=1),nn.BatchNorm2d(out_ch),nn.ReLU()) for _ in in_channels[:-1]])
    def forward(self, feats):
        lats=[self.lat[i](feats[i]) for i in range(4)]; p=self.mamba[3](lats[3]); outs=[p]
        for i in range(2,-1,-1):
            p=F.interpolate(p,size=lats[i].shape[2:],mode='bilinear',align_corners=False)
            p=self.mamba[i](self.fuse[i](p+lats[i])); outs.append(p)
        return outs

class QuantumHQCM(nn.Module):
    def __init__(self, nq=8, nl=4):
        super().__init__()
        self.nq=nq; self.dev=qml.device('default.qubit',wires=nq)
        @qml.qnode(self.dev,interface='torch',diff_method='backprop')
        def circuit(inputs,weights):
            qml.AngleEmbedding(inputs,wires=range(nq),rotation='Y')
            qml.StronglyEntanglingLayers(weights,wires=range(nq))
            return [qml.expval(qml.PauliZ(i)) for i in range(nq)]
        self.circuit=circuit
        self.weights=nn.Parameter(torch.randn(qml.StronglyEntanglingLayers.shape(n_layers=nl,n_wires=nq))*0.1)
        self.pre_ln=nn.LayerNorm(nq); self.post_ln=nn.LayerNorm(nq)
    def forward(self, x):
        orig_dtype, orig_device = x.dtype, x.device
        x=torch.tanh(self.pre_ln(x))*math.pi
        # PennyLane needs float64 on CPU
        x_cpu=x.detach().cpu().double(); w_cpu=self.weights.detach().cpu().double()
        results=[torch.stack(self.circuit(x_cpu[i],w_cpu)) for i in range(x_cpu.shape[0])]
        return self.post_ln(torch.stack(results).to(dtype=orig_dtype, device=orig_device))

class DummyHQCM(nn.Module):
    def __init__(self, nq=8):
        super().__init__()
        self.fc=nn.Sequential(nn.Linear(nq,nq*4),nn.GELU(),nn.Linear(nq*4,nq)); self.ln=nn.LayerNorm(nq)
    def forward(self,x): return self.ln(self.fc(x))

class SplineActivation(nn.Module):
    def __init__(self, n=8):
        super().__init__()
        self.knots=nn.Parameter(torch.linspace(-3,3,n)); self.coeffs=nn.Parameter(torch.zeros(n))
    def forward(self,x):
        d=x.unsqueeze(-1)-self.knots.view(1,1,1,-1)
        return x+(torch.exp(-0.5*d**2)*self.coeffs).sum(-1)

class QWaveKANHead(nn.Module):
    def __init__(self, in_ch=64, fd=8):
        super().__init__()
        self.q_gate=nn.Sequential(nn.Linear(fd,in_ch),nn.Sigmoid()); self.spline=SplineActivation()
        self.seg_head=nn.Conv2d(in_ch,1,1); self.uncert_head=nn.Conv2d(in_ch,1,1); self.edge_head=nn.Conv2d(in_ch,1,1)
    def forward(self, feat, fv):
        g=self.spline(feat*self.q_gate(fv)[:,:,None,None])
        return self.seg_head(g), self.uncert_head(g).abs(), self.edge_head(g)

class DualDomainFusion(nn.Module):
    def __init__(self, sc, fd=8):
        super().__init__()
        self.fe=nn.Sequential(nn.Linear(fd,sc),nn.Sigmoid())
        self.conv=nn.Sequential(nn.Conv2d(sc,sc,3,padding=1),nn.BatchNorm2d(sc),nn.ReLU())
    def forward(self, sp, fv): return self.conv(sp*self.fe(fv)[:,:,None,None]+sp)

class QFECOD(nn.Module):
    def __init__(self, pretrained=False, use_quantum=False):
        super().__init__()
        self.backbone=PVTv2Backbone(pretrained)
        ch=self.backbone.channels
        self.dwt=DWTDecomp(); self.fsa=FSA(ch[3])
        self.hqcm=QuantumHQCM() if use_quantum else DummyHQCM()
        self.fpn=FPNDecoder(ch,64); self.fusion=DualDomainFusion(64); self.head=QWaveKANHead()
    def forward(self, x):
        feats=self.backbone(x)
        fv=self.hqcm(self.fsa(*self.dwt(feats[3])))
        fine=self.fpn(feats)[-1]; fused=self.fusion(fine,fv)
        seg,unc,edge=self.head(fused,fv); H,W=x.shape[2:]
        return (F.interpolate(seg,(H,W),mode='bilinear',align_corners=False),
                F.interpolate(unc,(H,W),mode='bilinear',align_corners=False),
                F.interpolate(edge,(H,W),mode='bilinear',align_corners=False), fused)

class CAMClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone=timm.create_model('pvt_v2_b2',pretrained=False,features_only=False,num_classes=0)
        fd=self.backbone.num_features
        self.head=nn.Sequential(nn.Identity(),nn.Linear(fd,256),nn.ReLU(),nn.Dropout(0.3),nn.Linear(256,2))
    def forward(self,x): return self.head(self.backbone(x))

class SINetV2(nn.Module):
    def __init__(self, pretrained=False):
        super().__init__()
        bb=timm.create_model('resnet50',pretrained=pretrained,features_only=True,out_indices=(1,2,3,4))
        self.enc=bb; self.chs=bb.feature_info.channels()
        self.lat=nn.ModuleList([nn.Conv2d(c,64,1) for c in self.chs])
        self.sm=nn.ModuleList([nn.Sequential(nn.Conv2d(64,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU()) for _ in range(4)])
        self.im=nn.Sequential(nn.Conv2d(64,64,3,padding=1,groups=64),nn.Conv2d(64,64,1),nn.BatchNorm2d(64),nn.ReLU())
        self.seg_head=nn.Conv2d(64,1,1); self.edge_head=nn.Conv2d(64,1,1)
    def forward(self, x):
        feats=self.enc(x); lats=[self.lat[i](feats[i]) for i in range(4)]
        p=self.sm[3](lats[3])
        for i in range(2,-1,-1): p=self.sm[i](F.interpolate(p,size=lats[i].shape[2:],mode='bilinear',align_corners=False)+lats[i])
        p=self.im(p); H,W=x.shape[2:]
        seg=F.interpolate(self.seg_head(p),(H,W),mode='bilinear',align_corners=False)
        edge=F.interpolate(self.edge_head(p),(H,W),mode='bilinear',align_corners=False)
        return seg, torch.zeros_like(seg), edge, p

# ============================================================================
# CELL 4: Load Models
# ============================================================================
print('Loading models...')
use_quantum = torch.cuda.is_available()  # Only use quantum on CUDA

cls_model = CAMClassifier()
if (CKPT/'best_cls.pth').exists():
    cls_model.load_state_dict(torch.load(CKPT/'best_cls.pth', map_location='cpu'))
    print('  ✓ Classifier loaded')
cls_model = cls_model.to(DEVICE).eval()

qfe_model = QFECOD(pretrained=False, use_quantum=use_quantum)
if (CKPT/'best_qfe.pth').exists():
    qfe_model.load_state_dict(torch.load(CKPT/'best_qfe.pth', map_location='cpu'), strict=False)
    print(f'  ✓ QFE-COD loaded (quantum={use_quantum})')
qfe_model = qfe_model.to(DEVICE).eval()

sinet_model = SINetV2(pretrained=False)
if (CKPT/'best_sinet.pth').exists():
    sinet_model.load_state_dict(torch.load(CKPT/'best_sinet.pth', map_location='cpu'))
    print('  ✓ SINetV2 loaded')
sinet_model = sinet_model.to(DEVICE).eval()
print('All models ready.')

# ============================================================================
# CELL 5: Inference Functions
# ============================================================================
def preprocess(img_pil):
    img_np = np.array(img_pil.convert('RGB'))
    return val_aug(image=img_np)['image'].unsqueeze(0).float().to(DEVICE), img_np

@torch.no_grad()
def run_pipeline(image):
    if image is None: return (None,)*11
    pil = Image.fromarray(image) if isinstance(image, np.ndarray) else image
    inp, orig = preprocess(pil)

    # Classify
    logits = cls_model(inp)
    probs = F.softmax(logits, dim=1)[0].cpu().numpy()
    lbl = logits.argmax(1).item()
    cls_text = '🦎 Camouflaged' if lbl==1 else '🔵 Non-Camouflaged'
    cls_detail = f"Confidence: {probs[lbl]:.1%}\nCAM: {probs[1]:.4f} | NonCAM: {probs[0]:.4f}"

    # Cls chart
    fig,ax=plt.subplots(figsize=(4,2)); ax.barh(['NonCAM','CAM'],[probs[0],probs[1]],color=['#4facfe','#f5576c'])
    ax.set_xlim(0,1); ax.set_facecolor('#0d1117'); fig.patch.set_facecolor('#0d1117')
    ax.tick_params(colors='white')
    for l in ax.get_yticklabels(): l.set_color('white')
    for s in ax.spines.values(): s.set_color('#30363d')
    plt.tight_layout(); fig.canvas.draw()
    cls_chart=np.frombuffer(fig.canvas.buffer_rgba(),dtype=np.uint8).reshape(fig.canvas.get_width_height()[::-1]+(4,))[:,:,:3]
    plt.close(fig)

    # If Non-Camouflaged, skip segmentation
    if lbl == 0:
        return cls_text, cls_detail, cls_chart, None, None, None, None, None, None, None, None

    def run_seg(model):
        seg,unc,edge,_=model(inp)
        sp=torch.sigmoid(seg)[0,0].cpu().numpy(); ep=torch.sigmoid(edge)[0,0].cpu().numpy()
        un=unc[0,0].cpu().numpy()
        orig_r=cv2.resize(orig,(IMG_SIZE,IMG_SIZE)).astype(np.float32)/255.0
        # Heatmap overlay
        hm=cm.jet(sp)[:,:,:3]; ov=(0.55*hm+0.45*orig_r).clip(0,1)
        # Uncertainty
        un_n=(un-un.min())/(un.max()-un.min()+1e-8) if un.max()>0 else un
        # Use continuous probability maps (not binary) so they're visible
        seg_vis = (sp * 255).astype(np.uint8)            # grayscale probability
        edge_vis = (ep * 255).astype(np.uint8)            # grayscale probability
        return (seg_vis, edge_vis,
                (ov*255).astype(np.uint8), (cm.inferno(un_n)[:,:,:3]*255).astype(np.uint8), sp)

    qs,qe,qo,qu,qp = run_seg(qfe_model)
    ss,se,so,su,sp2 = run_seg(sinet_model)

    # Comparison
    fig,ax=plt.subplots(figsize=(4,2.5))
    m=['Coverage%','MeanConf%']
    qv=[(qp>0.5).mean()*100, qp[qp>0.5].mean()*100 if (qp>0.5).any() else 0]
    sv=[(sp2>0.5).mean()*100, sp2[sp2>0.5].mean()*100 if (sp2>0.5).any() else 0]
    x=np.arange(2); ax.bar(x-0.175,qv,0.35,label='QFE',color='#f5576c'); ax.bar(x+0.175,sv,0.35,label='SINet',color='#4facfe')
    ax.set_xticks(x); ax.set_xticklabels(m); ax.legend(fontsize=8,facecolor='#161b22',edgecolor='#30363d',labelcolor='white')
    ax.set_facecolor('#0d1117'); fig.patch.set_facecolor('#0d1117'); ax.tick_params(colors='white')
    for s in ax.spines.values(): s.set_color('#30363d')
    for l in ax.get_xticklabels()+ax.get_yticklabels(): l.set_color('white')
    plt.tight_layout(); fig.canvas.draw()
    cmp=np.frombuffer(fig.canvas.buffer_rgba(),dtype=np.uint8).reshape(fig.canvas.get_width_height()[::-1]+(4,))[:,:,:3]
    plt.close(fig)

    return cls_text, cls_detail, cls_chart, qs, qe, qo, qu, ss, se, so, cmp

# ============================================================================
# CELL 6: Gradio Dashboard
# ============================================================================
CSS = """
/* Dark premium theme */
.gradio-container {
    background: radial-gradient(circle at 50% 0%, #1a1a2e 0%, #0d1117 60%) !important;
    font-family: 'Inter', 'Segoe UI', sans-serif !important;
    color: #e6edf3 !important;
}
.dark {
    --background-fill-primary: rgba(13, 17, 23, 0.6) !important;
    --background-fill-secondary: rgba(22, 27, 34, 0.4) !important;
    --border-color-primary: rgba(48, 54, 61, 0.5) !important;
    --block-title-text-color: #e6edf3 !important;
    --block-background-fill: rgba(22, 27, 34, 0.5) !important;
}
footer { display: none !important; }

/* Glowing header */
h1 {
    background: linear-gradient(90deg, #ff0844 0%, #ffb199 50%, #4facfe 100%) !important;
    -webkit-background-clip: text !important;
    -webkit-text-fill-color: transparent !important;
    font-weight: 900 !important;
    text-align: center !important;
    font-size: 2.6rem !important;
    letter-spacing: -1px !important;
    text-shadow: 0 0 40px rgba(255, 8, 68, 0.3) !important;
    margin-bottom: 5px !important;
}
h2, h3 { color: #ffffff !important; font-weight: 700 !important; letter-spacing: 0.5px !important; }

/* Glassmorphism Cards */
.block {
    background: rgba(22, 27, 34, 0.4) !important;
    backdrop-filter: blur(12px) !important; -webkit-backdrop-filter: blur(12px) !important;
    border: 1px solid rgba(255, 255, 255, 0.08) !important;
    border-radius: 16px !important;
    box-shadow: 0 8px 32px rgba(0, 0, 0, 0.3) !important;
    transition: transform 0.3s ease, box-shadow 0.3s ease !important;
}
.block:hover {
    transform: translateY(-2px) !important;
    box-shadow: 0 12px 40px rgba(0, 0, 0, 0.4), 0 0 20px rgba(79, 172, 254, 0.1) !important;
    border: 1px solid rgba(255, 255, 255, 0.15) !important;
}

/* Neon Primary Button */
.primary {
    background: linear-gradient(135deg, #f5576c 0%, #f093fb 100%) !important;
    border: none !important; color: white !important; font-weight: 800 !important;
    text-transform: uppercase !important; letter-spacing: 1px !important;
    border-radius: 12px !important; padding: 12px 28px !important;
    transition: all 0.4s cubic-bezier(0.175, 0.885, 0.32, 1.275) !important;
    box-shadow: 0 8px 25px rgba(245, 87, 108, 0.4) !important;
}
.primary:hover {
    box-shadow: 0 12px 35px rgba(245, 87, 108, 0.6), 0 0 15px rgba(240, 147, 251, 0.5) !important;
    transform: scale(1.03) translateY(-2px) !important;
    background: linear-gradient(135deg, #ff6b80 0%, #f5a3ff 100%) !important;
}
.primary:active { transform: scale(0.98) translateY(1px) !important; }

/* Textbox & Inputs styling */
.textbox, .input {
    background: rgba(13, 17, 23, 0.7) !important; border: 1px solid rgba(48, 54, 61, 0.8) !important;
    color: #e6edf3 !important; border-radius: 10px !important; transition: all 0.3s ease !important;
}
.textbox:focus-within { border-color: #4facfe !important; box-shadow: 0 0 10px rgba(79, 172, 254, 0.3) !important; }

/* Tab Navigation */
.tab-nav { border-bottom: 1px solid rgba(255, 255, 255, 0.1) !important; margin-bottom: 20px !important; }
.tab-nav button {
    color: #8b949e !important; background: transparent !important; border: none !important;
    font-weight: 700 !important; font-size: 1rem !important; text-transform: uppercase !important;
    letter-spacing: 1px !important; padding: 12px 20px !important; transition: all 0.3s ease !important;
    position: relative !important;
}
.tab-nav button:hover { color: #e6edf3 !important; }
.tab-nav button.selected { color: #00f2fe !important; background: transparent !important; }
.tab-nav button.selected::after {
    content: '' !important; position: absolute !important; bottom: -1px !important; left: 0 !important;
    width: 100% !important; height: 3px !important;
    background: linear-gradient(90deg, #00f2fe, #4facfe) !important;
    border-radius: 3px 3px 0 0 !important; box-shadow: 0 -2px 10px rgba(0, 242, 254, 0.5) !important;
}

/* Image containers */
.image-container {
    border-radius: 12px !important; overflow: hidden !important;
    border: 1px solid rgba(255, 255, 255, 0.05) !important;
}
.image-container img { transition: transform 0.5s ease !important; }
.image-container:hover img { transform: scale(1.02) !important; }

/* Labels */
label, .label-wrap span {
    color: #a3b3c4 !important; font-weight: 700 !important;
    text-transform: uppercase !important; font-size: 0.75rem !important; letter-spacing: 1px !important;
}
"""

with gr.Blocks(css=CSS, title="QFE-COD Dashboard",
               theme=gr.themes.Base(primary_hue="red",neutral_hue="gray",font=gr.themes.GoogleFont("Inter"))) as app:
    gr.Markdown("# 🔬 QFE-COD • Quantum Camouflage Detection\n**PVTv2-B4 + DWT + FSA + Mamba + HQCM + Q-WaveKAN** vs **SINetV2** | COD10K-v3")

    with gr.Tabs():
        with gr.Tab("🔍 Inference"):
            gr.Markdown("### Upload an image to run all three models")
            with gr.Row():
                with gr.Column(scale=1):
                    inp_img = gr.Image(label="📷 Upload Image", type="numpy", height=352)
                    run_btn = gr.Button("⚡ Run Full Pipeline", variant="primary", size="lg")
                    gr.Markdown(f"<small style='color:#8b949e'>Device: **{DEVICE}** | Size: **{IMG_SIZE}²**</small>")
                with gr.Column(scale=1):
                    cls_res = gr.Textbox(label="Classification", lines=1, interactive=False)
                    cls_det = gr.Textbox(label="Details", lines=2, interactive=False)
                    cls_img = gr.Image(label="Probabilities", height=180)

            gr.Markdown("### 🧠 QFE-COD Predictions")
            with gr.Row():
                qseg=gr.Image(label="Seg Mask",height=250); qedge=gr.Image(label="Edge",height=250)
                qov=gr.Image(label="Heatmap",height=250); qunc=gr.Image(label="Uncertainty",height=250)

            gr.Markdown("### 📐 SINetV2 Predictions")
            with gr.Row():
                sseg=gr.Image(label="Seg Mask",height=250); sedge=gr.Image(label="Edge",height=250)
                sov=gr.Image(label="Heatmap",height=250)

            gr.Markdown("### 📊 Comparison")
            cmp_img = gr.Image(label="QFE vs SINet", height=220)

            run_btn.click(run_pipeline, [inp_img], [cls_res,cls_det,cls_img,qseg,qedge,qov,qunc,sseg,sedge,sov,cmp_img])

        with gr.Tab("🏗️ Architecture"):
            gr.Markdown("""### Pipeline
```
IMAGE → PVTv2-B4 → [F1-F4]
                     F4 → DWT → FSA → HQCM(8-qubit) → freq_vec
                     [F1-F4] → FPN+Mamba → P1
                     P1 + freq_vec → DualDomainFusion → Q-WaveKAN → SEG/UNCERT/EDGE
```
| Model | Params | Backbone |
|-------|--------|----------|
| Classifier | ~25M | PVTv2-B2 |
| QFE-COD | ~63M | PVTv2-B4 |
| SINetV2 | ~24M | ResNet-50 |""")

app.launch(share=True, prevent_thread_lock=True, show_error=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 73.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.6/935.6 kB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 79.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 105.6 MB/s eta 0:00:00
Checkpoints: [PosixPath('/kaggle/input/datasets/dakshethg/qfe-pred/checkpoints/last_sinet.pth'), PosixPath('/kaggle/input/datasets/dakshethg/qfe-pred/checkpoints/best_qfe.pth'), PosixPath('/kaggle/input/datasets/dakshethg/qfe-pred/checkpoints/best_cls.pth'), PosixPath('/kaggle/input/datasets/dakshethg/qfe-pred/checkpoints/best_sinet.pth'), PosixPath('/kaggle/input/datasets/dakshethg/qfe-pred/checkpoints/last_qfe.pth')]
Logs: [PosixPath('/kaggle/